In [ ]:
"""
Week 2 - Day 1
DQN Architecture
=================
Building Deep Q-Network to replace
Q-table for dynamic pricing.

Why DQN?
→ Q-table limited to small state spaces
→ Neural Network scales to any state space
→ Can generalize across similar states!

Internship Spec:
"replace the Q-table with a Neural Network
(Deep Q-Network)"

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import torch
import matplotlib.pyplot as plt
import os
import sys

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

SRC_DIR = os.path.join(PROJECT_ROOT, "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Current Working Directory:", os.getcwd())
print("Project Root:", PROJECT_ROOT)
print("SRC Directory:", SRC_DIR)
print("SRC exists:", os.path.exists(SRC_DIR))

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.dqn.dqn_network import (
    DQNNetwork,
    DuelingDQNNetwork
)
from agents.dqn.replay_buffer import ReplayBuffer
from agents.dqn.dqn_agent import DQNAgent
from config import DQN

plt.style.use('seaborn-v0_8')
print("✅ DQN modules loaded!")
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

In [ ]:
# Create and display network
net = DQNNetwork(
    state_size=2,
    action_size=6,
    hidden_sizes=[128, 64]
)
net.print_architecture()

print(f"\nNetwork Structure:")
print(net)

In [ ]:
# Test with sample states
test_states = [
    [50, 30],   # Full inventory, lots of time
    [50, 5],    # Full inventory, urgent!
    [10, 30],   # Low inventory, lots of time
    [10, 5],    # Low inventory, urgent!
    [25, 15],   # Half and half
]

print("=== Q-VALUES FOR KEY STATES ===\n")
for state in test_states:
    # Normalize
    norm = [state[0]/50, state[1]/30]
    tensor = torch.FloatTensor([norm])

    with torch.no_grad():
        q_vals = net(tensor).numpy()[0]

    best_action = q_vals.argmax()
    best_price  = PRICE_LEVELS[best_action]

    print(f"  State ({state[0]:2d} inv, {state[1]:2d} days):")
    print(f"  Q-values: {q_vals.round(3)}")
    print(f"  Best price: ${best_price}\n")

print("Note: Network not trained yet!")
print("Q-values are random (untrained weights)")

In [ ]:
# Demonstrate replay buffer
buffer = ReplayBuffer(capacity=1000)

print("Filling replay buffer...")
env = DynamicPricingEnv()
obs, _ = env.reset(seed=42)

# Fill with random experiences
for _ in range(500):
    action = env.action_space.sample()
    next_obs, reward, term, trunc, info = (
        env.step(action)
    )
    buffer.push(obs, action, reward, next_obs,
                term or trunc)
    if term or trunc:
        obs, _ = env.reset()
    else:
        obs = next_obs

print(f"\nBuffer filled!")
buffer.print_stats()

# Sample a batch
states, actions, rewards, next_states, dones = (
    buffer.sample(32)
)
print(f"\nSampled batch of 32:")
print(f"  States      : {states.shape}")
print(f"  Actions     : {actions.shape}")
print(f"  Rewards     : {rewards.shape}")
print(f"  Mean reward : ${rewards.mean():.0f}")

In [ ]:
print("Quick DQN training test (200 episodes)...")
print("Just to verify everything works!\n")

env   = DynamicPricingEnv()
agent = DQNAgent(env, DQN)

# Short training
rewards = agent.train(
    n_episodes=200,
    verbose=True
)

print(f"\n✅ DQN training works!")
print(f"   Final mean revenue: "
      f"${np.mean(rewards[-50:]):.0f}")

In [ ]:
# Compare standard vs dueling DQN
standard = DQNNetwork(2, 6, [128, 64])
dueling  = DuelingDQNNetwork(2, 6, 128)

std_params = sum(
    p.numel() for p in standard.parameters()
)
duel_params = sum(
    p.numel() for p in dueling.parameters()
)

print("=== ARCHITECTURE COMPARISON ===\n")
print(f"  Standard DQN  : {std_params} parameters")
print(f"  Dueling DQN   : {duel_params} parameters")
print(f"\n  Standard: Input → 128 → 64 → 6")
print(f"  Dueling : Input → 128 → (V:64→1) + "
      f"(A:64→6) → 6")
print(f"\n  Week 2: Start with Standard DQN")
print(f"  Week 3: Can upgrade to Dueling DQN!")

In [ ]:
print("╔══════════════════════════════════════════╗")
print("║    WEEK 2 DAY 1 — DQN ARCHITECTURE      ║")
print("╠══════════════════════════════════════════╣")
print("║  COMPONENTS BUILT:                       ║")
print("║  ✅ DQN Neural Network (2→128→64→6)     ║")
print("║  ✅ Dueling DQN Architecture             ║")
print("║  ✅ Experience Replay Buffer             ║")
print("║  ✅ Prioritized Replay Buffer            ║")
print("║  ✅ Complete DQN Agent skeleton          ║")
print("╠══════════════════════════════════════════╣")
print("║  KEY INNOVATIONS:                        ║")
print("║  → Neural Network replaces Q-table      ║")
print("║  → Experience replay breaks correlations║")
print("║  → Target network stabilizes training   ║")
print("║  → Epsilon greedy for exploration       ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Full DQN Training! 🚀       ║")
print("╚══════════════════════════════════════════╝")